# Feast feature store on oblako

[Feast](https://feast.dev) is an open-source **feature store**. Its Redshift
*offline store* is a client of two AWS surfaces only: the `redshift-data` API
(to run SQL) and S3 (for `UNLOAD`/`COPY` of Parquet). oblako serves both, so the
**exact same Feast code you'd run on AWS** works here — only the endpoints change.

This notebook runs the full lifecycle against oblako:

| Feast step | Backed by |
|---|---|
| offline store (historical features, materialization source) | **redshift-local** (`redshift-data` → wire proxy → PostgreSQL) |
| `UNLOAD`/`COPY` staging | the proxy's in-image **S3 bridge** → **S3Proxy** |
| online store (low-latency serving) | **DynamoDB Local** |
| registry | a local file |

Start the services first:

```bash
docker compose up -d redshift s3proxy dynamodb
```

and install the client deps: `pip install 'feast[aws]' redshift_connector`.

## 1. Point boto3 at oblako

Unmodified Feast/boto3 code; only `AWS_ENDPOINT_URL_*` differ from AWS. (In the oblako notebook kernel these are already set for you.)

In [ ]:
import os

os.environ["AWS_ACCESS_KEY_ID"] = "test"
os.environ["AWS_SECRET_ACCESS_KEY"] = "test"
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
os.environ["AWS_ENDPOINT_URL_S3"] = "http://localhost:9000"
os.environ["AWS_ENDPOINT_URL_REDSHIFT_DATA"] = "http://localhost:8002"
os.environ["AWS_ENDPOINT_URL_DYNAMODB"] = "http://localhost:8001"
# S3Proxy needs path-style addressing and no SDK-v2 flexible checksums.
os.environ["AWS_REQUEST_CHECKSUM_CALCULATION"] = "when_required"
os.environ["AWS_S3_ADDRESSING_STYLE"] = "path"

## 2. Seed the source table and staging bucket

Feast reads features from a Redshift table. We create one through the `redshift-data`
API (real SQL against redshift-local) and make the S3 staging bucket.

In [ ]:
import time

import boto3

rsd = boto3.client("redshift-data")


def run_sql(sql):
    sid = rsd.execute_statement(Database="oblako", ClusterIdentifier="oblako", Sql=sql)[
        "Id"
    ]
    while True:
        d = rsd.describe_statement(Id=sid)
        if d["Status"] in ("FINISHED", "FAILED", "ABORTED"):
            break
        time.sleep(0.1)
    assert d["Status"] == "FINISHED", d.get("Error")
    return sid


run_sql("DROP TABLE IF EXISTS driver_hourly_stats")
run_sql("""CREATE TABLE driver_hourly_stats (
    driver_id INTEGER, event_timestamp TIMESTAMP, conv_rate REAL,
    acc_rate REAL, avg_daily_trips INTEGER, created TIMESTAMP)""")
run_sql("""INSERT INTO driver_hourly_stats VALUES
    (1001, TIMESTAMP '2021-04-12 10:59:42', 0.1, 0.5, 10, now()),
    (1001, TIMESTAMP '2021-04-12 08:12:10', 0.2, 0.6, 20, now()),
    (1002, TIMESTAMP '2021-04-12 10:59:42', 0.3, 0.7, 30, now()),
    (1003, TIMESTAMP '2021-04-12 10:59:42', 0.4, 0.8, 40, now())""")

try:
    boto3.client("s3").create_bucket(Bucket="feast-staging")
except Exception as e:
    print("bucket:", e)
print("source table + staging bucket ready")

## 3. Define the feature store (offline = redshift, online = dynamodb)

We build the config in code (no YAML), so the notebook is self-contained.

> Re-running `apply` against an already-created table errors, because DynamoDB Local
> doesn't support tag reconciliation; restart the `dynamodb` service to run from scratch.

In [ ]:
from datetime import timedelta

from feast import Entity, FeatureView, Field, FeatureStore, RepoConfig
from feast.infra.offline_stores.redshift import RedshiftOfflineStoreConfig
from feast.infra.offline_stores.redshift_source import RedshiftSource
from feast.infra.online_stores.dynamodb import DynamoDBOnlineStoreConfig
from feast.types import Float32, Int64

driver = Entity(name="driver", join_keys=["driver_id"])

source = RedshiftSource(
    name="driver_hourly_stats_source",
    table="driver_hourly_stats",
    timestamp_field="event_timestamp",
    created_timestamp_column="created",
)

driver_hourly_stats = FeatureView(
    name="driver_hourly_stats",
    entities=[driver],
    ttl=timedelta(days=3650),
    schema=[
        Field(name="conv_rate", dtype=Float32),
        Field(name="acc_rate", dtype=Float32),
        Field(name="avg_daily_trips", dtype=Int64),
    ],
    online=True,
    source=source,
)

config = RepoConfig(
    project="oblako_feast",
    provider="aws",
    registry="/tmp/oblako_feast/registry.db",
    offline_store=RedshiftOfflineStoreConfig(
        cluster_id="oblako",
        region="us-east-1",
        user="oblako",
        database="oblako",
        s3_staging_location="s3://feast-staging/staging",
        iam_role="arn:aws:iam::000000000000:role/oblako",
    ),
    online_store=DynamoDBOnlineStoreConfig(
        region="us-east-1", endpoint_url="http://localhost:8001"
    ),
    entity_key_serialization_version=2,
)

store = FeatureStore(config=config)
store.apply([driver, driver_hourly_stats])
print("applied:", [fv.name for fv in store.list_feature_views()])

## 4. Historical features (point-in-time join)

`get_historical_features` stages the entity dataframe to S3, runs the point-in-time
join in redshift-local, and `UNLOAD`s the result — all through the in-image S3 bridge.

In [ ]:
import pandas as pd

entity_df = pd.DataFrame(
    {
        "driver_id": [1001, 1002, 1003],
        "event_timestamp": [pd.Timestamp("2021-04-12 12:00:00")] * 3,
    }
)

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
    ],
).to_df()
training_df
# driver 1001 -> conv_rate 0.1: the 10:59 row, the latest at or before 12:00.

## 5. Materialize to the online store, then serve

`materialize` loads the latest feature values into **DynamoDB Local**; `get_online_features`
reads them back with low latency — the serving path.

In [ ]:
from datetime import datetime, timezone

store.materialize(
    start_date=datetime(2021, 1, 1, tzinfo=timezone.utc),
    end_date=datetime(2021, 12, 31, tzinfo=timezone.utc),
)

online = store.get_online_features(
    features=[
        "driver_hourly_stats:conv_rate",
        "driver_hourly_stats:acc_rate",
        "driver_hourly_stats:avg_daily_trips",
    ],
    entity_rows=[{"driver_id": 1001}, {"driver_id": 1002}, {"driver_id": 1003}],
).to_dict()
pd.DataFrame(online)

## Recap

The same Feast code that targets managed Redshift + DynamoDB ran unchanged against
oblako: **redshift-local** as the offline store (with the wire proxy bridging
`UNLOAD`/`COPY` to **S3Proxy**), and **DynamoDB Local** as the online store. Point the
endpoints back at AWS and it runs there — nothing else changes.